In [68]:
from mlc.cashflow import ScorableModelTemplate, compute_score
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline

from xgboost import XGBClassifier


In [ ]:
transaction_df = pd.read_parquet("transactions.parquet")
transaction_df = transaction_df.dropna()
transaction_df['category'] = transaction_df['category'].astype(int)

consumer_df = pd.read_parquet("consumer_data.parquet")
categories_df = pd.read_csv("transaction_categories.csv")

consumer_df['evaluation_date'] = pd.to_datetime(consumer_df['evaluation_date'], unit = 'ns')
transaction_df['posted_date'] = pd.to_datetime(transaction_df['posted_date'], unit = 'ns')



In [3]:
transactions_df = pd.merge(transaction_df,consumer_df[['masked_consumer_id', 'evaluation_date']],on='masked_consumer_id',how='left')
transactions_df = transactions_df[transactions_df['posted_date'] <= transactions_df['evaluation_date']]

### Feature engineering

In [ ]:
def count_positive(series):
    return (series > 0).sum()

def count_negative(series):
    return (series < 0).sum()

cat_gb = transaction_df.groupby(by=['masked_consumer_id', 'category'])['amount'].agg(mean='mean', median='median',max='max', min='min', positive_count = count_positive, negative_count = count_negative)
cat_gb = cat_gb.reset_index()

In [49]:
features = ['mean','positive_count','negative_count', 'max','min']

catf_df = cat_gb.melt(
    id_vars = ['masked_consumer_id', 'category'],
    value_vars = features
).pivot_table(
    index=['masked_consumer_id'],
    columns= ['category', 'variable']
)


catf_df=catf_df.reset_index()
catf_df.columns = ['_'.join(str(level) for level in col) for col in catf_df.columns]
catf_df = catf_df.set_index('masked_consumer_id__')

chosen_features = ['value_20_negative_count', 'value_5_mean', 'value_15_negative_count',
       'value_10_min', 'value_5_positive_count', 'value_8_positive_count',
       'value_1_negative_count', 'value_7_positive_count',
       'value_34_negative_count', 'value_10_max', 'value_3_mean',
       'value_16_negative_count', 'value_10_mean', 'value_27_negative_count',
       'value_26_negative_count', 'value_14_negative_count',
       'value_18_negative_count', 'value_13_negative_count',
       'value_22_negative_count', 'value_3_positive_count']

catf_df = catf_df[chosen_features]


In [44]:
catf_df

,value_20_negative_count,value_5_mean,value_15_negative_count,value_10_min,value_5_positive_count,value_8_positive_count,value_1_negative_count,value_7_positive_count,value_34_negative_count,value_10_max,value_3_mean,value_16_negative_count,value_10_mean,value_27_negative_count,value_26_negative_count,value_14_negative_count,value_18_negative_count,value_13_negative_count,value_22_negative_count,value_3_positive_count
0,131.0,1726.005556,62.0,NaN,9.0,12.0,210.0,15.0,NaN,NaN,1603.681429,533.0,NaN,94.0,72.0,361.0,102.0,13.0,56.0,21.0
1,62.0,726.122308,86.0,NaN,13.0,NaN,142.0,24.0,1.0,NaN,844.427742,290.0,NaN,52.0,102.0,591.0,279.0,41.0,86.0,31.0
2,14.0,NaN,80.0,NaN,NaN,NaN,8.0,24.0,12.0,NaN,1651.705429,20.0,NaN,4.0,10.0,141.0,51.0,NaN,36.0,33.0
3,8.0,NaN,164.0,NaN,NaN,24.0,182.0,2.0,NaN,NaN,NaN,68.0,NaN,29.0,14.0,109.0,168.0,24.0,4.0,NaN
4,50.0,NaN,40.0,NaN,NaN,NaN,506.0,7.0,1.0,NaN,NaN,181.0,NaN,64.0,66.0,495.0,153.0,32.0,31.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15874,21.0,652.502381,NaN,NaN,21.0,NaN,40.0,NaN,NaN,NaN,700.717500,227.0,NaN,9.0,17.0,208.0,96.0,NaN,1.0,4.0
15875,48.0,NaN,17.0,NaN,NaN,NaN,152.0,NaN,NaN,NaN,3382.241538,375.0,NaN,21.0,NaN,162.0,78.0,20.0,24.0,13.0
15876,94.0,NaN,NaN,NaN,NaN,NaN,55.0,NaN,NaN,NaN,3392.631111,314.0,NaN,33.0,17.0,666.0,324.0,27.0,35.0,27.0
15877,55.0,NaN,57.0,NaN,NaN,22.0,198.0,NaN,4.0,NaN,492.263636,229.0,NaN,26.0,34.0,279.0,60.0,15.0,65.0,22.0


In [39]:
catf_df

,value_20_negative_count,value_5_mean,value_15_negative_count,value_10_min,value_5_positive_count,value_8_positive_count,value_1_negative_count,value_7_positive_count,value_34_negative_count,value_10_max,value_3_mean,value_16_negative_count,value_10_mean,value_27_negative_count,value_26_negative_count,value_14_negative_count,value_18_negative_count,value_13_negative_count,value_22_negative_count,value_3_positive_count
0,131.0,1726.005556,62.0,NaN,9.0,12.0,210.0,15.0,NaN,NaN,1603.681429,533.0,NaN,94.0,72.0,361.0,102.0,13.0,56.0,21.0
1,62.0,726.122308,86.0,NaN,13.0,NaN,142.0,24.0,1.0,NaN,844.427742,290.0,NaN,52.0,102.0,591.0,279.0,41.0,86.0,31.0
2,14.0,NaN,80.0,NaN,NaN,NaN,8.0,24.0,12.0,NaN,1651.705429,20.0,NaN,4.0,10.0,141.0,51.0,NaN,36.0,33.0
3,8.0,NaN,164.0,NaN,NaN,24.0,182.0,2.0,NaN,NaN,NaN,68.0,NaN,29.0,14.0,109.0,168.0,24.0,4.0,NaN
4,50.0,NaN,40.0,NaN,NaN,NaN,506.0,7.0,1.0,NaN,NaN,181.0,NaN,64.0,66.0,495.0,153.0,32.0,31.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15874,21.0,652.502381,NaN,NaN,21.0,NaN,40.0,NaN,NaN,NaN,700.717500,227.0,NaN,9.0,17.0,208.0,96.0,NaN,1.0,4.0
15875,48.0,NaN,17.0,NaN,NaN,NaN,152.0,NaN,NaN,NaN,3382.241538,375.0,NaN,21.0,NaN,162.0,78.0,20.0,24.0,13.0
15876,94.0,NaN,NaN,NaN,NaN,NaN,55.0,NaN,NaN,NaN,3392.631111,314.0,NaN,33.0,17.0,666.0,324.0,27.0,35.0,27.0
15877,55.0,NaN,57.0,NaN,NaN,22.0,198.0,NaN,4.0,NaN,492.263636,229.0,NaN,26.0,34.0,279.0,60.0,15.0,65.0,22.0


In [34]:
dates_df = transaction_df.groupby(by='masked_consumer_id')['posted_date'].min()
dates_df = pd.merge(dates_df, consumer_df[['masked_consumer_id', 'evaluation_date']], on='masked_consumer_id')
dates_df['diff_date'] = (dates_df['evaluation_date']-dates_df['posted_date']).dt.days
dates_df = dates_df.set_index('masked_consumer_id')['diff_date']

Combining all features

In [50]:
features_df = consumer_df.set_index('masked_consumer_id')\
    .join(dates_df, how='left')\
    .join(catf_df, how='left')

### Split into categories and train test split

In [97]:
features_df['consumer_category'] = features_df.index.str[2].astype(int)
features_df = pd.get_dummies(features_df, columns=['consumer_category'], prefix='Category', dtype=int)

In [100]:
features_df_arr = [0]*4
for i in range(4):
    features_df_arr[i] = features_df[features_df[f'Category_{i+1}'] == 1].copy()

In [117]:
# Training without splitting categories at all. Measure over all dataset
X = features_df.drop(columns=['evaluation_date','FPF_TARGET'])
y = features_df['FPF_TARGET']
X_train, X_val, y_train, y_val = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)
pipeline = Pipeline([('model', XGBClassifier(n_estimators=50,learning_rate=0.05, random_state=42))])
pipeline.fit(X_train,y_train)
y_train_pred = pipeline.predict_proba(X_train)[:, 1]
y_val_pred = pipeline.predict_proba(X_val)[:, 1]
train_auc = roc_auc_score(y_train, y_train_pred)
val_auc = roc_auc_score(y_val, y_val_pred)
print(f'Train AUC: {train_auc:.3f}, Validation AUC: {val_auc:.3f}')


Train AUC: 0.940, Validation AUC: 0.883


In [107]:
# Split into categories, train and measure
for i,df in enumerate(features_df_arr):
    X = df.drop(columns=['evaluation_date','FPF_TARGET'])
    y = df['FPF_TARGET']
    X_train, X_val, y_train, y_val = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)
    pipeline = Pipeline([('model', XGBClassifier(n_estimators=40,learning_rate=0.03, random_state=42))])
    pipeline.fit(X_train,y_train)
    y_train_pred = pipeline.predict_proba(X_train)[:, 1]
    y_val_pred = pipeline.predict_proba(X_val)[:, 1]
    train_auc = roc_auc_score(y_train, y_train_pred)
    val_auc = roc_auc_score(y_val, y_val_pred)
    print(f'Category: {i+1}')
    print(f'Train AUC: {train_auc:.3f}, Validation AUC: {val_auc:.3f}')


Category: 1
Train AUC: 0.917, Validation AUC: 0.629
Category: 2
Train AUC: 0.874, Validation AUC: 0.603
Category: 3
Train AUC: 0.922, Validation AUC: 0.653
Category: 4
Train AUC: 0.942, Validation AUC: 0.559


In [116]:
# Train, split test into categories, measure.
X = features_df.drop(columns=['evaluation_date','FPF_TARGET'])
y = features_df['FPF_TARGET']
X_train, X_val, y_train, y_val = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)
pipeline = Pipeline([('model', XGBClassifier(n_estimators=50,learning_rate=0.05, random_state=42))])
pipeline.fit(X_train,y_train)
y_train_pred = pipeline.predict_proba(X_train)[:, 1]
y_val_pred = pipeline.predict_proba(X_val)[:, 1]
for i in range(1,5):
    y_train_i = y_train[X_train[f'Category_{i}']==1]
    y_val_i = y_val[X_val[f'Category_{i}']==1]
    y_train_pred_i = y_train_pred[X_train[f'Category_{i}']==1]
    y_val_pred_i = y_val_pred[X_val[f'Category_{i}']==1]
    train_auc = roc_auc_score(y_train_i, y_train_pred_i)
    val_auc = roc_auc_score(y_val_i, y_val_pred_i)
    print(f'Category: {i}')
    print(f'Train AUC: {train_auc:.3f}, Validation AUC: {val_auc:.3f}')



Category: 1
Train AUC: 0.874, Validation AUC: 0.591
Category: 2
Train AUC: 0.860, Validation AUC: 0.606
Category: 3
Train AUC: 0.897, Validation AUC: 0.730
Category: 4
Train AUC: 0.863, Validation AUC: 0.615
